<a href="https://colab.research.google.com/github/chaupham31251020816-png/teamwork/blob/main/banknotes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
import os

train_dir = '/content/extracted_money/Vietnamese banknotes'

ImageDataGenerator = tf.keras.preprocessing.image.ImageDataGenerator
train_datagen = ImageDataGenerator(rescale=1.0/255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(200, 200),
    batch_size=4,
    class_mode='categorical',
    shuffle=True
)

class_names = list(train_generator.class_indices.keys())
num_classes = len(class_names)
print("💵 Các mệnh giá AI quét được và chuẩn bị học:", class_names)

model = Sequential([
    Conv2D(32, (3, 3), activation="relu", input_shape=(200, 200, 3)),
    MaxPooling2D(2, 2),
    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D(2, 2),
    Flatten(),
    Dense(128, activation="relu"),
    Dense(num_classes, activation="softmax")
])

model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

print("🚀 AI bắt đầu học thuộc lòng dữ liệu tiền...")
model.fit(train_generator, epochs=15)

model.save('/content/money_model.h5')
with open('/content/money_classes.txt', 'w') as f:
    for name in class_names:
        f.write(name + '\n')

print("\n🎉 ĐÃ ÉP LƯU FILE MODEL CHUẨN THÀNH CÔNG!")
print("Danh sách nhãn mô hình đã Master thuộc lòng:", class_names)

Found 35 images belonging to 6 classes.
💵 Các mệnh giá AI quét được và chuẩn bị học: ['.ipynb_checkpoints', '100k', '1k', '2k', '500k', '5k']
🚀 AI bắt đầu học thuộc lòng dữ liệu tiền...
Epoch 1/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 8s 731ms/step - accuracy: 0.0857 - loss: 11.7475
Epoch 2/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 9s 862ms/step - accuracy: 0.3429 - loss: 1.7461
Epoch 3/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 6s 626ms/step - accuracy: 0.4286 - loss: 1.5405
Epoch 4/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 8s 895ms/step - accuracy: 0.5714 - loss: 1.2229
Epoch 5/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 6s 622ms/step - accuracy: 0.8286 - loss: 0.7250
Epoch 6/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 6s 686ms/step - accuracy: 0.8857 - loss: 0.4234
Epoch 7/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 7s 779ms/step - accuracy: 0.9429 - loss: 0.2938
Epoch 8/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 6s 630ms/step - accuracy: 0.8286 - loss: 0.7262
Epoch 9/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 8s 890ms/step - accuracy: 0.9143 - loss: 0.4291
Epoch 10/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 6s 641ms/step - accu


🎉 ĐÃ ÉP LƯU FILE MODEL CHUẨN THÀNH CÔNG!
Danh sách nhãn mô hình đã Master thuộc lòng: ['.ipynb_checkpoints', '100k', '1k', '2k', '500k', '5k']


In [33]:
!pip install pyngrok streamlit -q

with open("app_money.py", "w", encoding="utf-8") as f:
    f.write("""
import streamlit as st
import tensorflow as tf
import numpy as np
from PIL import Image
import os

st.set_page_config(page_title="Vietnamese Banknotes AI", layout="centered")
st.title("💵 Ứng dụng Nhận diện Tiền Việt Nam bằng AI")

def load_money_model():
    if os.path.exists('/content/money_model.h5'):
        return tf.keras.models.load_model('/content/money_model.h5')
    return None

def load_money_labels():
    if os.path.exists('/content/money_classes.txt'):
        with open('/content/money_classes.txt', 'r') as f:
            return [line.strip() for line in f.readlines()]
    return None

class_names = load_money_labels()
model = load_money_model()

if model is None or class_names is None:
    st.error("❌ Không tìm thấy file model.h5 hoặc nhãn lớp. Hãy chạy ô Train trước!")
else:
    st.success("✅ Hệ thống AI đã sẵn sàng nhận diện!")
    option = st.radio("Chọn cách quét tiền:", ("Tải ảnh từ máy", "Đưa tiền vào Camera"))
    image_file = st.file_uploader("Chọn file ảnh tiền", type=["jpg", "jpeg", "png"]) if option == "Tải ảnh từ máy" else st.camera_input("Đưa tờ tiền trước camera")

    if image_file is not None:
        img = Image.open(image_file)
        st.image(img, caption='Ảnh đầu vào', use_column_width=True)
        img_resized = img.resize((200, 200))
        img_array = np.array(img_resized)
        if img_array.shape[-1] == 4:
            img_array = img_array[:, :, :3]

        img_tensor = np.expand_dims(img_array, axis=0).astype('float32') / 255.0
        preds = model.predict(img_tensor)
        idx = np.argmax(preds[0])
        confidence = preds[0][idx] * 100

        pred_money = class_names[idx]

        display_label = pred_money
        if 'k' in pred_money.lower():
            val = pred_money.lower().replace('k', '')
            if val.isdigit(): display_label = f"{int(val):,}k"
        else:
            if pred_money.isdigit(): display_label = f"{int(pred_money):,}"

        st.success(f"### Mệnh giá dự đoán: **{display_label} VNĐ**")
        st.info(f"Độ chính xác: **{confidence:.2f}%**")
""")

print("✅ Đã ghi đè file app_money.py!")

from pyngrok import ngrok
import time

!fuser -k 8501/tcp
!pkill -9 streamlit
!pkill -9 ngrok
ngrok.kill()

ngrok.set_auth_token("3DCzkYVyk7cztKhBEp7gOn1Yinv_31y5SnP1LeyZkPoADqfK1")

get_ipython().system_raw("streamlit run app_money.py --server.port 8501 --server.address 127.0.0.1 &")
print("⌛ Đang nổ máy hệ thống mạng, đợi 15 giây...")
time.sleep(15)

try:
    public_url = ngrok.connect("127.0.0.1:8501", bind_tls=True)
    print("\n=======================================================")
    print("🎉 KHỞI TẠO APP THÀNH CÔNG:")
    print(f"👉 Link mở Web ứng dụng ở đây: {public_url.public_url}")
    print("=======================================================")
except Exception as e:
    print(f"\n❌ Lỗi tạo link mạng: {e}")

✅ Đã ghi đè file app_money.py!
8501/tcp:            17868
⌛ Đang nổ máy hệ thống mạng, đợi 15 giây...

🎉 KHỞI TẠO APP THÀNH CÔNG:
👉 Link mở Web ứng dụng ở đây: https://evaluator-clamshell-aftermost.ngrok-free.dev
